In [36]:
import pandas as pd
import spacy

In [37]:
df = pd.read_csv('../data/events/2_struct/2000-2025.csv')

df.head()

,date_start,date_end,event
0,2000-01-01,NaN,Деноминация белорусского рубля;
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н..."
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог..."
3,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...


In [38]:
import spacy
from spacy.matcher import DependencyMatcher

nlp = spacy.load("ru_core_news_lg")
dm = DependencyMatcher(nlp.vocab)

# 1️⃣ Ввели / наложили санкции
pattern_action = [
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"LEMMA": {"IN": ["ввести", "наложить"]}}},
    {"LEFT_ID": "verb", "REL_OP": ">>", "RIGHT_ID": "sanction", "RIGHT_ATTRS": {"LEMMA": "санкция"}},
]

lemmas_pkg = [
    "принять", "принят", "принятие",
    "утвердить", "утвержден", "утверждение",
    "одобрить", "одобрен", "одобрение"
]

pattern_pkg_any = [
    {
        "RIGHT_ID": "root",
        "RIGHT_ATTRS": {
            "LEMMA": {"IN": lemmas_pkg}
        },
    },
    {
        "LEFT_ID": "root",
        "REL_OP": ">>",  # допускаем промежуточные звенья (числительные, прилагательные)
        "RIGHT_ID": "package",
        "RIGHT_ATTRS": {"LEMMA": "пакет"},
    },
    {
        "LEFT_ID": "package",
        "REL_OP": ">>",
        "RIGHT_ID": "sanction",
        "RIGHT_ATTRS": {"LEMMA": "санкция"},
    },
]


dm.add("SANCTION", [pattern_action, pattern_pkg_any])

# === Тест ===
text = """
Принят 9 пакет санкций против России.
"""

doc = nlp(text)

matches = dm(doc)
seen = set()
for mid, toks in matches:
    span = doc[min(toks): max(toks)+1]
    key = tuple(sorted(toks))
    if key in seen:
        continue
    seen.add(key)
    print(f"{nlp.vocab.strings[mid]} → {span.text}")

SANCTION → Принят 9 пакет санкций


In [40]:
def has_sanction(text, matcher=dm):
    doc = nlp(text)
    matches = matcher(doc)
    return len(matches) > 0

df["is_sanction"] = df["event"].apply(has_sanction)

print(len(df[df["is_sanction"] == True]))
df

17


,date_start,date_end,event,is_sanction
0,2000-01-01,NaN,Деноминация белорусского рубля;,False
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н...",False
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог...",False
3,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...,False
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...,False
...,...,...,...,...
5642,2025-09-18,NaN,на Камчатке зафиксировано землетрясение магнит...,False
5643,2025-09-20,NaN,проведение конкурса песни «Интервидение» в Мос...,False
5644,2025-09-23,NaN,Международный уголовный суд представил подтвер...,False
5645,2025-09-25,NaN,Парламент Кыргызстана объявил о самороспуске.,False
